In [4]:
pip install langchain

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install langchain_groq

   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   --------------------------------------- 548.1/548.1 kB 19.1 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 1.4.0
    Uninstalling groq-1.4.0:
      Successfully uninstalled groq-1.4.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.7
    Uninstalling langchain-core-1.2.7:
      Successfully uninstalled langchain-core-1.2.7
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [94]:
!pip install tavily-python


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


In [97]:
import os
from duckduckgo_search import DDGS

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from tavily import TavilyClient

In [107]:
GROQ_API_KEY = "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"
TAVILY_API_KEY = "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"

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
client = TavilyClient(api_key=TAVILY_API_KEY)

In [108]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.4
)

In [109]:
HUMAN_IN_LOOP = False

In [ ]:
class NewsletterAgent:

    def __init__(self, goal, human_mode=False):

        self.goal = goal
        self.human_mode = human_mode

        self.articles = []
        self.summaries = []

        self.newsletter_md = ""
        self.newsletter_html = ""

    # =========================================
    # STEP 1 — PLAN
    # =========================================

    def plan(self):

        print("\n================================")
        print("[STEP 1] PLANNING")
        print("================================\n")

        steps = [
            "Research latest AI news",
            "Collect relevant articles",
            "Summarize articles",
            "Generate newsletter",
            "Critique newsletter",
            "Improve final output",
            "Generate HTML version",
            "Save outputs",
            "Simulate email sending"
        ]

        for step in steps:
            print(f"✓ {step}")

    # =========================================
    # STEP 2 — RESEARCH
    # =========================================

    def research_news(self):

        print("\n================================")
        print("[STEP 2] RESEARCHING NEWS")
        print("================================\n")

        response = client.search(
            query="""
            Latest AI agent news from OpenAI, Anthropic,
            Google DeepMind, Microsoft AI, autonomous AI agents
            """,
            search_depth="advanced",
            max_results=7
        )

        articles = []

        for result in response["results"]:

            articles.append({
                "title": result["title"],
                "body": result["content"],
                "link": result["url"]
            })

        self.articles = articles

        print(f"✓ Found {len(articles)} trusted articles\n")

        for i, article in enumerate(articles):

            print(f"{i+1}. {article['title']}")
            print(article['link'])
            print()

        if self.human_mode:

            approve = input(
                "\nApprove these articles? (yes/no): "
            )

            if approve.lower() != "yes":

                print("Process stopped by user.")
                exit()

    # =========================================
    # STEP 3 — SUMMARIZE
    # =========================================

    def summarize_articles(self):

        print("\n================================")
        print("[STEP 3] SUMMARIZING ARTICLES")
        print("================================\n")

        summaries = []

        prompt = PromptTemplate(
            input_variables=["title", "body"],
            template="""
You are an AI newsletter writer.

Summarize this AI news article.

TITLE:
{title}

ARTICLE:
{body}

Return:
1. Summary
2. Why it matters
3. Key takeaway
"""
        )

        for article in self.articles:

            chain = prompt | llm

            response = chain.invoke({
                "title": article["title"],
                "body": article["body"]
            })

            summaries.append({
                "title": article["title"],
                "summary": response.content,
                "link": article["link"]
            })

        self.summaries = summaries

        print(f"✓ Summarized {len(summaries)} articles")

    # =========================================
    # STEP 4 — GENERATE NEWSLETTER
    # =========================================

    def generate_newsletter(self):

        print("\n================================")
        print("[STEP 4] GENERATING NEWSLETTER")
        print("================================\n")

        content = ""

        for article in self.summaries:

            content += f"""
# {article['title']}

{article['summary']}

Source:
{article['link']}

---

"""

        newsletter_prompt = PromptTemplate(
            input_variables=["content"],
            template="""
Create a professional AI newsletter.

Requirements:
- Attractive title
- Introduction
- Professional formatting
- Reader friendly structure
- Conclusion
- Markdown formatting

STRICT RULES: 
- ONLY use provided researched articles 
- DO NOT invent news 
- DO NOT invent URLs 
- DO NOT invent experts 
- DO NOT invent quotes 
- DO NOT invent interviews 
- DO NOT invent events 
- DO NOT use placeholder text 
- DO NOT mention fake infographics 
- ONLY use real article links 
- Keep tone professional and factual 
- Modern AI newsletter style

CONTENT:
{content}
"""
        )

        chain = newsletter_prompt | llm

        response = chain.invoke({
            "content": content
        })

        self.newsletter_md = response.content

        print("✓ Newsletter generated")

    # =========================================
    # STEP 5 — CRITIQUE
    # =========================================

    def critique_newsletter(self):

        print("\n================================")
        print("[STEP 5] SELF CRITIQUE")
        print("================================\n")

        critique_prompt = PromptTemplate(
            input_variables=["newsletter"],
            template="""
Review this newsletter briefly.

Check:
- readability
- clarity
- engagement
- formatting
- repetition
- missing insights

Return ONLY: 1. One strength 2. One weakness 3. One improvement suggestion Keep response under 80 words.

NEWSLETTER:
{newsletter}
"""
        )

        chain = critique_prompt | llm

        critique = chain.invoke({
            "newsletter": self.newsletter_md
        })

        print("\nAI CRITIQUE:\n")
        print(critique.content)

        improve_prompt = PromptTemplate(
            input_variables=["newsletter", "critique"],
            template="""
Improve this newsletter using the critique.

NEWSLETTER:
{newsletter}

CRITIQUE:
{critique}
"""
        )

        improve_chain = improve_prompt | llm

        improved = improve_chain.invoke({
            "newsletter": self.newsletter_md,
            "critique": critique.content
        })

        self.newsletter_md = improved.content

        print("\n✓ Newsletter improved")


    # =========================================
    # STEP 6 — HTML
    # =========================================

    def generate_html(self):

        print("\n================================")
        print("[STEP 6] GENERATING HTML")
        print("================================\n")

        html = markdown2.markdown(self.newsletter_md)

        self.newsletter_html = f"""
                                    <html>
                                    <body>
                                    {html}
                                    </body>
                                    </html>
                                    """

        print("✓ HTML generated")

    # =========================================
    # STEP 7 — SAVE
    # =========================================

    def save_outputs(self):

        print("\n================================")
        print("[STEP 7] SAVING FILES")
        print("================================\n")

        with open("newsletter.md", "w", encoding="utf-8") as f:
            f.write(self.newsletter_md)

        with open("newsletter.html", "w", encoding="utf-8") as f:
            f.write(self.newsletter_html)

        print("✓ Files saved")

    # =========================================
    # STEP 8 — EMAIL
    # =========================================

    def simulate_email(self):

        print("\n================================")
        print("[STEP 8] EMAIL SIMULATION")
        print("================================\n")

        print("Subject: Weekly AI Newsletter\n")

        print(self.newsletter_md[:1500])

        print("\n✓ Email simulated")

    # =========================================
    # MAIN RUN METHOD
    # =========================================

    def run(self):

        print("\n================================")
        print("AUTONOMOUS NEWSLETTER AGENT")
        print("================================")

        self.plan()

        self.research_news()

        self.summarize_articles()

        self.generate_newsletter()

        self.critique_newsletter()

        self.generate_html()

        self.save_outputs()

        self.simulate_email()

        print("\n================================")
        print("AGENT EXECUTION COMPLETED")
        print("================================")

In [111]:
def run_newsletter_agent(goal, human_mode=False):

    agent = NewsletterAgent(
        goal=goal,
        human_mode=human_mode
    )

    agent.run()

In [112]:
import markdown2 
import markdown

In [85]:
pip install -U ddgs

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


In [113]:
from ddgs import DDGS

In [114]:
goal = "Create a weekly newsletter on latest AI agent news and send it to subscribers."

run_newsletter_agent(
    goal=goal,
    human_mode=False
)


AUTONOMOUS NEWSLETTER AGENT

[STEP 1] PLANNING

✓ Research latest AI news
✓ Collect relevant articles
✓ Summarize articles
✓ Generate newsletter
✓ Critique newsletter
✓ Improve final output
✓ Generate HTML version
✓ Save outputs
✓ Simulate email sending

[STEP 2] RESEARCHING NEWS

✓ Found 7 trusted articles

1. Google DeepMind launches Deep Research Max autonomous AI research agent | ETIH EdTech News — EdTech Innovation Hub
https://www.edtechinnovationhub.com/news/googles-new-ai-agent-will-run-160-searches-while-you-sleep-and-hand-you-the-report-by-morning

2. What’s Next in AI: Five Trends to Watch in 2026
https://blog.bytebytego.com/p/whats-next-in-ai-five-trends-to-watch

3. Google AI announcements from April 2026
https://blog.google/innovation-and-ai/technology/ai/google-ai-updates-april-2026

4. What's next in AI: 7 trends to watch in 2026 - Microsoft Source
https://news.microsoft.com/source/features/ai/whats-next-in-ai-7-trends-to-watch-in-2026

5. Microsoft Build 2025: The age 

In [115]:
import os

print(os.listdir())

['app.py', 'final_newsletter.ipynb', 'newsletter.html', 'newsletter.md', 'requirements.txt', '__pycache__']
